In [ ]:
# ipython -c "%run plot.ipynb"

import matplotlib
from matplotlib import gridspec
import matplotlib.pyplot as plt
from matplotlib import style
import pandas as pd
import numpy as np
from pathlib import Path

# Paper specific settings
STANDARD_WIDTH = 17.8
SINGLE_COL_WIDTH = STANDARD_WIDTH / 2
DOUBLE_COL_WIDTH = STANDARD_WIDTH
def cm_to_inch(value):
    return value / 2.54

# matplotlib style settings
matplotlib.rcParams['text.usetex'] = False
style.use('bmh')
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.grid.axis'] = 'y'
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['hatch.linewidth'] = 0.5
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['grid.linestyle'] = '--'
pd.options.display.max_columns = None
pd.options.display.max_rows = None

# Data preprocessing: this notebook uses AgentTX's recorded VM measurements.
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
history = pd.read_csv(RESULTS / 'motivation_optimization_history.csv')
runtime = pd.read_csv(RESULTS / 'motivation_runtime_comparison.csv')
history_full = history[history['metric'] == 'full_ms_per_step'].copy()
history_full['label'] = ['trace bypass', 'read effects', 'script reuse', 'defer GC', 'direct script', 'persistent worker']

# Plotting settings
fig = plt.figure(dpi=300, figsize=(cm_to_inch(DOUBLE_COL_WIDTH), cm_to_inch(6.2)))
cmap = 'tab10'
patterns = ['///', '']
bar_width = 0.34

# (a) Historical before/after measurements
ax0 = plt.subplot(1, 2, 1)
x = np.arange(len(history_full))
before = history_full['before'].astype(float).to_numpy()
after = history_full['after'].astype(float).to_numpy()
bars_before = ax0.bar(x - bar_width / 2, before, width=bar_width, hatch=patterns[0], color=plt.get_cmap(cmap)(0), linewidth=0.5, label='before')
bars_after = ax0.bar(x + bar_width / 2, after, width=bar_width, hatch=patterns[1], color=plt.get_cmap(cmap)(3), linewidth=0.5, label='after')
ax0.set_xticks(x, labels=['W/D\ntrace', 'READ/NEG\neffects', 'script\nreuse', 'defer\nGC', 'direct\nscript', 'persistent\nworker'], fontsize=6)
ax0.set_ylabel('Full AgentTX latency (ms/step)', fontsize=8)
ax0.tick_params(bottom=False, top=False, left=False, right=False)
ax0.tick_params(axis='y', labelsize=8)
ax0.set_title('(a) Optimization chain', fontsize=8)
ax0.legend(loc='upper right', fontsize=6, frameon=False, handlelength=1.2, columnspacing=0.5)

# (b) Current execution baselines
ax1 = plt.subplot(1, 2, 2)
runtime_order = ['bare', 'per_call_try', 'shared_try', 'shared_checkpoint', 'agenttx_without_read_tracing', 'agenttx_full']
runtime_names = ['bare', 'per-call\ntry', 'shared\ntry', 'shared\ncheckpoint', 'AgentTX\nno-trace', 'AgentTX\nfull']
runtime = runtime.set_index('mode').loc[runtime_order].reset_index()
values = runtime['per_step_mean_ms'].astype(float).to_numpy()
bar_colors = [plt.get_cmap(cmap)(0), plt.get_cmap(cmap)(1), plt.get_cmap(cmap)(1), plt.get_cmap(cmap)(2), plt.get_cmap(cmap)(4), plt.get_cmap(cmap)(3)]
bars = ax1.bar(np.arange(len(values)), values, width=0.62, color=bar_colors, linewidth=0.5, hatch=['', '///', '///', '', '', ''])
ax1.set_xticks(np.arange(len(values)), labels=runtime_names, fontsize=6)
ax1.set_ylabel('Latency (ms/step)', fontsize=8)
ax1.tick_params(bottom=False, top=False, left=False, right=False)
ax1.tick_params(axis='y', labelsize=8)
ax1.set_title('(b) Current baselines', fontsize=8)
for bar, value in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width() / 2, value + max(values) * 0.02, f'{value:.1f}', ha='center', va='bottom', fontsize=6, rotation=90)

for ax in fig.axes:
    lw = 0.5
    for axis in ['top', 'bottom', 'left', 'right']:
        ax.spines[axis].set_linewidth(lw)

plt.tight_layout(pad=0.4)
plt.savefig(ROOT / 'motivation' / 'FIG-Motivation-Optimization.pdf', bbox_inches='tight', pad_inches=0)
plt.show()
